In [39]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd

from bayes_opt import BayesianOptimization, acquisition
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("future.no_silent_downcasting", True)
warnings.filterwarnings("ignore")

## Import and Prepare Data

**Select either  `removal` or `recovery` as the desired Output here!**

After selecting the desired Output, the required data is imported. For simplicity, the features (Flow_C, Flow_P, and Current_Limit_P) are always imported from the cache folder here, which saves having to convert them to the correct format from the CSV files. Depending on which output variable you selected previously, either the data for the output variable *Average CO2 Removal* or *Average Water Recovery* is loaded. The scaler used in the data preparation step is always imported i inverse transform the results.

In [40]:
output = "removal"

In [41]:
cached_objects_data_path = os.path.join(
    os.path.dirname(os.getcwd()), "..", "cached_objects"
)

X = joblib.load(os.path.join(cached_objects_data_path, "prepared_data", "X.npy"))

if output == "removal":
    y = joblib.load(
        os.path.join(cached_objects_data_path, "prepared_data", "y_removal.npy")
    ).reshape(-1, 1)
    y_label = joblib.load(
        os.path.join(cached_objects_data_path, "prepared_data", "y_label_removal.npy")
    )
    y_name = "removal"
elif output == "recovery":
    y = joblib.load(
        os.path.join(cached_objects_data_path, "prepared_data", "y_recovery.npy")
    ).reshape(-1, 1)
    y_label = joblib.load(
        os.path.join(cached_objects_data_path, "prepared_data", "y_label_recovery.npy")
    )
    y_name = "recovery"

scaler: MinMaxScaler = joblib.load(
    os.path.join(cached_objects_data_path, "prepared_data", "scaler.pkl")
)

## Suggest next Measurement

Bayesian Optimization has already been used as a method for optimization, but here we will show another use for it. For optimization, we used the *maximize* function, which feeds the parameter sets calculated by the *Acquisition Function* directly back into the function to be optimized. Here, we want to output precisely this calculated value in order to start a targeted next test run.

The structure is almost exactly the same as for optimizing the black box function, except that there is no longer a black box function which uses one of the models. Instead, an acquisition function is provided to the optimizer directly during creation. When creating the acquisition function, a parameter Kappa is provided. The lower this parameter is, the more the already registered measured values are included in the calculation.

The suggested values for the next test run are shown below.

In [42]:
# Prepare acquisition function
acq_func = acquisition.UpperConfidenceBound(kappa=2.5, exploration_decay=None, exploration_decay_delay=None)

# Specify the maximum and minimum values to be probed by the optimizer
pbounds = {
    "Flow_C": (0.0, X[:, 0].max()),
    "Flow_P": (0.0, X[:, 1].max()),
    "Current_Limit_P": (0.0, X[:, 2].max()),
}

# Create optimizer
optimizer = BayesianOptimization(
    f=None,
    acquisition_function=acq_func,
    pbounds=pbounds,
    verbose=0,
    random_state=1,
    allow_duplicate_points=False,
)

# Register input data
excluded_output_vals = [0.99]
registered_points = set()
for i in range(len(X)):
    if y[i, 0] not in excluded_output_vals:
        point = (X[i, 0], X[i, 1], X[i, 2])
        if point not in registered_points:
            optimizer.register(
                params={
                    "Flow_C": X[i, 0],
                    "Flow_P": X[i, 1],
                    "Current_Limit_P": X[i, 2],
                },
                target=y[i, 0],
            )
            registered_points.add(point)

# Suggest the next parameter
next_to_probe = optimizer.suggest()

# Transform data to its original value range
inputs = np.array(list(next_to_probe.values()))
best_x_original = scaler.inverse_transform(inputs.reshape(1, -1))[0]

print("Next Flow_C", best_x_original[0])
print("Next Flow_P:", best_x_original[1])
print("Next Current_Limit_P:", best_x_original[2])

Next Flow_C 0.6989366161067736
Next Flow_P: 0.5250382781628388
Next Current_Limit_P: 50.00249424484753
